# 02 - Detecciones basadas en una plantilla

## Descripcion

Este notebook genera predicciones sobre la base de datos basadas en la similitud con una plantilla. 
Primero carga la base de datos de embeddings, luego genera y compara los embeddings de una plantilla usada como ejemplo para seleccionar las señales similares. Se realiza una anotación rápida de los 50 señales con mejor correspodencia y se utiliza estas anotaciones para crear un modelo iniciar que predice sobre la base de datos los segmentos de audio más similares a la plantilla. Se guarda estas predicciones en un csv con información sobre el score de cada segmento de audio. 

### Importar dependencias

In [ ]:
import os

from matplotlib import pyplot as plt
import numpy as np

from perch_hoplite.agile import audio_loader
from perch_hoplite.agile import classifier
from perch_hoplite.agile import classifier_data
from perch_hoplite.agile import embedding_display
from perch_hoplite.agile import source_info
from perch_hoplite.db  import brutalism
from perch_hoplite.db import score_functions
from perch_hoplite.db  import search_results
from perch_hoplite.db import sqlite_usearch_impl
from perch_hoplite.zoo import model_configs
from perch_hoplite.zoo import taxonomy_model_tf

### Definir la ruta de la hoplite db (embeddings) que se va a utilizar

In [ ]:
db_path = '/mnt/d/Taboga/Train/perch_embed' 

### Cargar database

In [ ]:
# Load model and connect to database {vertical-output: true}
# Identifier (e.g. name) to attach to labels produced during validation.
annotator_id = 'linnaeus'

db = sqlite_usearch_impl.SQLiteUSearchDB.create(db_path)
db_model_config = db.get_metadata('model_config')
embed_config = db.get_metadata('audio_sources')
model_class = model_configs.get_model_class(db_model_config.model_key)
embedding_model = model_class.from_config(db_model_config.model_config)
audio_sources = source_info.AudioSources.from_config_dict(embed_config)
if hasattr(embedding_model, 'window_size_s'):
  window_size_s = embedding_model.window_size_s
else:
  window_size_s = 5.0
audio_filepath_loader = audio_loader.make_filepath_loader(
    audio_sources=audio_sources,
    window_size_s=window_size_s,
    sample_rate_hz=embedding_model.sample_rate,
)


# corregir un problema de compatibilidad de numpy
import numpy as np
from perch_hoplite.db import sqlite_usearch_impl

def get_embeddings_batch_fixed(self, window_ids):
    embeddings_batch = self.ui.get(window_ids)

    if isinstance(embeddings_batch, tuple):
        embeddings_batch = np.stack(embeddings_batch)

    if not isinstance(embeddings_batch, np.ndarray):
        raise RuntimeError(
            f"Expected np.ndarray or tuple, got {type(embeddings_batch)}"
        )

    return embeddings_batch

sqlite_usearch_impl.SQLiteUSearchDB.get_embeddings_batch = get_embeddings_batch_fixed


ids = brutalism.get_brute_search_ids(db, sample_size=10, rng_seed=42)
emb = db.get_embeddings_batch(ids)

print(type(emb))
print(emb.shape)

## Busqueda

### Cargar la plantilla para consulta (query)
El `query_uri` puede ser una URL, una ruta de archivo o un ID de Xeno-Canto (como `xc105133`, que contiene un zorzal de bosque (`woothr`)).

In [ ]:
query_uri = '/mnt/d/Taboga/Cebus01.wav'  
query_label = 'cebimi' # Nombre de la anotacion

query = embedding_display.QueryDisplay(
    uri=query_uri, offset_s=0.0, window_size_s=5.0, sample_rate_hz=32000)
_ = query.display_interactive()

Visualizar resultados

In [ ]:
# Embed the Query and Search

# numero de resultados a mostrar
num_results = 10 
query_embedding = embedding_model.embed(
    query.get_audio_window()).embeddings[0, 0]

# If checked, search for examples near a particular target score.
target_sampling = False  # @param {type: 'boolean'}

# When target sampling, target this score.
target_score = -1.0  # @param
if not target_sampling:
  target_score = None

# If True, search the full DB. Otherwise, use approximate nearest-neighbor search.
exact_search = False 

results = db.search(
    query_embedding,
    search_list_size=num_results,
    approximate=exact_search,
    target_score=target_score,
)
# Get a random batch of scores to plot the score distribution.
scores = brutalism.get_random_embedding_scores(
    db, query_embedding, score_fn=score_functions.get_score_fn('dot'),
    sample_size=2_048,
    rng_seed=42,
)
_ = plt.hist(scores, bins=25, density=True, alpha=0.5)
hit_scores = [r.sort_score for r in results.search_results]
plt.scatter(hit_scores, np.zeros_like(hit_scores), marker='|',
            color='r', alpha=0.5)


### Realizar anotaciones

Revisar y anotar las detecciones. Utilizar el boton bajo cada audio, al presionar varias veces el boton genera una anotacion diferente. Verde: positivo
Naranja: negativo
Gris: duda

También se puede dejar sin anotación. De hecho, en caso de duda es recomendable dejar el audio sin anotación para que no genere "ruido" en el modelo. 

In [ ]:
display_results = embedding_display.EmbeddingDisplayGroup.from_search_results(
    results,
    db,
    sample_rate_hz=32000,
    frame_rate=100,
    audio_loader=audio_filepath_loader,
)
display_results.display(positive_labels=[query_label],  paged_mode = False)

In [ ]:
# @title Save data labels {vertical-output: true}

print("Annotations before saving new labels:", len(db.get_all_annotations()))

db.insert_annotations(
    display_results.harvest_labels(annotator_id),
    handle_duplicates="skip",
)

print("Annotations after saving new labels:", len(db.get_all_annotations()))

### Generar modelo inicial para clasificacion

In [ ]:
# @title Classifier training {vertical-output: true}

# @markdown Set of labels to classify. If None, auto-populated from the DB.
target_labels = None  # @param

# @markdown Classifier traning hyperparams. These should not require tuning.
learning_rate = 1e-3  # @param
weak_neg_weight = 0.05  # @param
l2_mu = 0.000  # @param
num_steps = 128  # @param

train_ratio = 0.9  # @param
batch_size = 128  # @param
weak_negatives_batch_size = 128  # @param
loss_fn_name = 'bce'  # @param ['hinge', 'bce']

data_manager = classifier_data.AgileDataManager(
    target_labels=target_labels,
    db=db,
    train_ratio=train_ratio,
    min_eval_examples=1,
    batch_size=batch_size,
    weak_negatives_batch_size=weak_negatives_batch_size,
    rng=np.random.default_rng(seed=5))
print('Training for target labels : ')
print(data_manager.get_target_labels())
linear_classifier, eval_scores = classifier.train_linear_classifier(
    data_manager=data_manager,
    learning_rate=learning_rate,
    weak_neg_weight=weak_neg_weight,
    num_train_steps=num_steps,
)
print('\n' + '-' * 80)
top1 = eval_scores['top1_acc']
print(f'top-1      {top1:.3f}')
rocauc = eval_scores['roc_auc']
print(f'roc_auc    {rocauc:.3f}')
cmap = eval_scores['cmap']
print(f'cmap       {cmap:.3f}')



In [ ]:
# Nombrar y guardar el modelo inicial, se guarda en la direccion de la DB hoplite
nombre_modelo = 'agile_classifier_inicial.pt'
linear_classifier.save(os.path.join(db_path, nombre_modelo))

## Usar modelo para prediccion inicial

## Configuracion de parametros

Definir la ruta donde se encuentra el modelo a utilizar

In [ ]:
# Se puede usar la ruta del modelo recien guardado
model_path = os.path.join(db_path, nombre_modelo)
# o escribir directamente la ruta al modelo, ejemplo: model_path = '/mnt/d/Taboga/agile_classifier_inicial.pt'

print(f'El modelo a utilizar es {model_path}')

Definir el nombre de la etiqueta que se va a buscar, la definida en query_label

In [ ]:
target_label = 'cebimi'  # o usar target_label = query_label
print(f'Se buscará la etiqueta {target_label}')

In [ ]:
# Cargar modelo lineal
custom_classifier = classifier.LinearClassifier.load(model_path)

#### Grafico de predicciones

Este grafico sirve para evaluar la calidad de la predicción. Muestra las predicciones en color rojo. Si las predicciones estan bien separadas a la derecha, lejos de la mayoria de segmentos significa que se realizo una buena clasificacion. 

In [ ]:
# Revisar las predicciones 

num_results = 200 

#target_label_idx = (0)
target_label_idx = custom_classifier.classes.index(target_label)
class_query = custom_classifier.beta[:, target_label_idx]
bias = custom_classifier.beta_bias[target_label_idx]

#@markdown Number of (randomly selected) database entries to search over.
sample_size = 1_000_000  #@param

#@markdown Whether to use margin-sampling. If checked, search for examples
#@markdown with logits near a particular target score (usually 0).
margin_sampling = False  #@param {type: 'boolean'}

# @markdown When margin sampling, target this logit.
margin_target_score = -0.0  # @param
if not margin_sampling:
  margin_target_score = None
score_fn = score_functions.get_score_fn(
    'dot', bias=bias, target_score=margin_target_score)
results = brutalism.threaded_brute_search(
    db, class_query, num_results, score_fn=score_fn,
    sample_size=sample_size)

# Get a random batch of scores to plot the score distribution.
scores = brutalism.get_random_embedding_scores(
    db,
    class_query,
    score_fn=score_functions.get_score_fn('dot', bias=bias),
    sample_size=2_048,
    rng_seed=42,
)
plt.hist(scores, bins=25, density=True, alpha=0.5)
hit_scores = [r.sort_score for r in results.search_results]
_ = plt.scatter(hit_scores, np.zeros_like(hit_scores), marker='|',
            color='r', alpha=0.5)

#### Realizar la prediccion y guardar los resultados

##### Definir el nombre del archivo resultado

In [ ]:
from pathlib import Path

pred_name = "pred_inicial"

model_path = Path(model_path)
model_name = model_path.stem
model_dir = model_path.parent

output_recording_summary = (
    model_dir
    / f"{model_name}_{pred_name}_recording_summary.csv"
)

print(f"Modelo: {model_name}")
print(f"Directorio del modelo: {model_dir}")
print(
    "El resumen por grabación se guardará en:",
    output_recording_summary,
)

In [ ]:
# ============================================================
# Inferencia por lotes y resumen de scores por grabación
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import json
import sqlite3

import numpy as np
import pandas as pd
from tqdm.auto import tqdm


# ------------------------------------------------------------
# Configuración
# ------------------------------------------------------------

TARGET_LABEL = query_label

# Número de embeddings procesados simultáneamente.
# Puede reducirse a 1024 si hay problemas de memoria.
BATCH_SIZE = 4096

# Número de mejores ventanas que se utilizarán para calcular
# el promedio de scores altos de cada grabación.
TOP_K_WINDOWS = 5

# Referencia descriptiva para contar ventanas con logit >= 0.
# Esto NO es todavía el threshold final del clasificador.
REFERENCE_LOGIT = 0.0


# ------------------------------------------------------------
# Verificar que la clase está presente en el clasificador
# ------------------------------------------------------------

if TARGET_LABEL not in custom_classifier.classes:
    raise ValueError(
        f"La etiqueta '{TARGET_LABEL}' no está presente en "
        f"el clasificador. Clases disponibles: "
        f"{custom_classifier.classes}"
    )

target_label_idx = custom_classifier.classes.index(
    TARGET_LABEL
)

print("Etiqueta objetivo:", TARGET_LABEL)
print("Índice de la etiqueta:", target_label_idx)


# ------------------------------------------------------------
# Obtener todos los IDs de ventanas con embedding
# ------------------------------------------------------------

window_ids = np.asarray(
    db.match_window_ids(),
    dtype=np.int64,
)

if len(window_ids) == 0:
    raise ValueError(
        "La base de datos no contiene ventanas con embeddings."
    )

if len(np.unique(window_ids)) != len(window_ids):
    raise ValueError(
        "Se encontraron window_id duplicados."
    )

print(
    f"Ventanas que serán evaluadas: {len(window_ids):,}"
)


# ------------------------------------------------------------
# Inferencia por lotes
# ------------------------------------------------------------

window_scores = np.empty(
    len(window_ids),
    dtype=np.float32,
)

for start in tqdm(
    range(0, len(window_ids), BATCH_SIZE),
    desc="Calculando logits",
):

    end = min(
        start + BATCH_SIZE,
        len(window_ids),
    )

    batch_ids = window_ids[start:end]

    batch_embeddings = db.get_embeddings_batch(
        batch_ids
    )

    batch_logits = np.asarray(
        custom_classifier(batch_embeddings)
    )

    if batch_logits.ndim == 1:
        # Caso excepcional: salida unidimensional.
        batch_target_scores = batch_logits
    else:
        batch_target_scores = batch_logits[
            :,
            target_label_idx,
        ]

    if len(batch_target_scores) != len(batch_ids):
        raise ValueError(
            "El número de logits no coincide con el "
            "número de ventanas del lote."
        )

    window_scores[start:end] = batch_target_scores


window_predictions = pd.DataFrame({
    "window_id": window_ids,
    "score": window_scores,
})

print("\nResumen general de logits:")
print(window_predictions["score"].describe())

In [ ]:
# ============================================================
# Obtener metadata de ventanas y grabaciones desde SQLite
# ============================================================

sqlite_path = Path(db_path) / "hoplite.sqlite"

if not sqlite_path.exists():
    raise FileNotFoundError(
        f"No se encontró la base SQLite: {sqlite_path}"
    )

with sqlite3.connect(sqlite_path) as conn:

    window_metadata = pd.read_sql_query(
        """
        SELECT
            w.id AS window_id,
            w.recording_id,
            r.filename,
            r.deployment_id
        FROM windows AS w
        INNER JOIN recordings AS r
            ON w.recording_id = r.id
        """,
        conn,
    )

    all_recordings = pd.read_sql_query(
        """
        SELECT
            id AS recording_id,
            filename,
            deployment_id
        FROM recordings
        """,
        conn,
    )


# Verificaciones
if window_metadata["window_id"].duplicated().any():
    raise ValueError(
        "La tabla windows contiene window_id duplicados."
    )

if all_recordings["recording_id"].duplicated().any():
    raise ValueError(
        "La tabla recordings contiene recording_id duplicados."
    )


# Unir scores y metadata
predictions = window_predictions.merge(
    window_metadata,
    on="window_id",
    how="left",
    validate="one_to_one",
)

if predictions["recording_id"].isna().any():
    missing_ids = predictions.loc[
        predictions["recording_id"].isna(),
        "window_id",
    ].head(20).tolist()

    raise ValueError(
        "Algunas ventanas con embedding no aparecen en "
        f"SQLite. Primeros IDs: {missing_ids}"
    )

print(
    "Grabaciones representadas en las predicciones:",
    predictions["recording_id"].nunique(),
)

print(
    "Grabaciones registradas en SQLite:",
    len(all_recordings),
)

In [ ]:
# ============================================================
# Resumen de scores por grabación
# ============================================================

group_columns = [
    "recording_id",
    "filename",
    "deployment_id",
]


# ------------------------------------------------------------
# Estadísticos generales
# ------------------------------------------------------------

recording_summary = (
    predictions
    .groupby(
        group_columns,
        as_index=False,
        dropna=False,
    )
    .agg(
        n_windows=("window_id", "size"),

        preliminary_max_score=(
            "score",
            "max",
        ),

        preliminary_mean_score=(
            "score",
            "mean",
        ),

        preliminary_median_score=(
            "score",
            "median",
        ),

        preliminary_score_sd=(
            "score",
            "std",
        ),

        preliminary_n_above_reference=(
            "score",
            lambda x: int(
                (x >= REFERENCE_LOGIT).sum()
            ),
        ),
    )
)


recording_summary[
    "preliminary_score_sd"
] = (
    recording_summary[
        "preliminary_score_sd"
    ]
    .fillna(0.0)
)


recording_summary[
    "preliminary_fraction_above_reference"
] = (
    recording_summary[
        "preliminary_n_above_reference"
    ]
    / recording_summary["n_windows"]
)

In [ ]:
# ------------------------------------------------------------
# Promedio de las TOP_K_WINDOWS mejores ventanas por grabación
# ------------------------------------------------------------

top_k_rows = (
    predictions
    .sort_values(
        ["recording_id", "score"],
        ascending=[True, False],
    )
    .groupby(
        "recording_id",
        sort=False,
        group_keys=False,
    )
    .head(TOP_K_WINDOWS)
)


top_k_summary = (
    top_k_rows
    .groupby(
        "recording_id",
        as_index=False,
    )
    .agg(
        preliminary_top_k_mean_score=(
            "score",
            "mean",
        ),

        preliminary_top_k_min_score=(
            "score",
            "min",
        ),

        preliminary_n_top_k_windows=(
            "window_id",
            "size",
        ),
    )
)


recording_summary = recording_summary.merge(
    top_k_summary,
    on="recording_id",
    how="left",
    validate="one_to_one",
)

In [ ]:
# ------------------------------------------------------------
# Identificar la mejor ventana de cada grabación
# ------------------------------------------------------------

best_row_indices = (
    predictions
    .groupby("recording_id")["score"]
    .idxmax()
)


best_windows = (
    predictions
    .loc[
        best_row_indices,
        [
            "recording_id",
            "window_id",
            "score",
        ],
    ]
    .rename(
        columns={
            "window_id": "preliminary_best_window_id",
            "score": "preliminary_best_window_score",
        }
    )
)


recording_summary = recording_summary.merge(
    best_windows,
    on="recording_id",
    how="left",
    validate="one_to_one",
)

In [ ]:
if not np.allclose(
    recording_summary["preliminary_max_score"],
    recording_summary[
        "preliminary_best_window_score"
    ],
):
    raise ValueError(
        "El score de la mejor ventana no coincide "
        "con el score máximo de la grabación."
    )

In [ ]:
# ------------------------------------------------------------
# Percentil 95 del score de cada grabación
# ------------------------------------------------------------

recording_p95 = (
    predictions
    .groupby("recording_id")["score"]
    .quantile(0.95)
    .rename("preliminary_p95_score")
    .reset_index()
)


recording_summary = recording_summary.merge(
    recording_p95,
    on="recording_id",
    how="left",
    validate="one_to_one",
)

In [ ]:
# ============================================================
# Verificación de cobertura
# ============================================================

missing_recordings = (
    set(all_recordings["recording_id"])
    - set(recording_summary["recording_id"])
)

if missing_recordings:
    raise ValueError(
        f"{len(missing_recordings)} grabaciones no tienen "
        "ventanas con embeddings. Primeros IDs: "
        f"{sorted(missing_recordings)[:20]}"
    )


if len(recording_summary) != len(all_recordings):
    raise ValueError(
        "El número de grabaciones del resumen no coincide "
        "con el número de grabaciones registrado en SQLite."
    )

print(
    f"Resumen generado para "
    f"{len(recording_summary):,} grabaciones."
)

### Guardar

In [ ]:
# ============================================================
# Ordenar y guardar el resumen
# ============================================================

# Score principal recomendado para ordenar las grabaciones.
SELECTION_SCORE = "preliminary_top_k_mean_score"


recording_summary = (
    recording_summary
    .sort_values(
        [
            SELECTION_SCORE,
            "preliminary_max_score",
        ],
        ascending=[False, False],
    )
    .reset_index(drop=True)
)


recording_summary[
    "preliminary_recording_rank"
] = np.arange(
    1,
    len(recording_summary) + 1,
)


# Información sobre cómo se generó el resumen
recording_summary["target_label"] = TARGET_LABEL
recording_summary["top_k_windows"] = TOP_K_WINDOWS
recording_summary["reference_logit"] = REFERENCE_LOGIT


output_recording_summary = Path(
    output_recording_summary
)

output_recording_summary.parent.mkdir(
    parents=True,
    exist_ok=True,
)


recording_summary.to_csv(
    output_recording_summary,
    index=False,
)


print(
    "\nResumen por grabación guardado en:"
)

print(output_recording_summary)

print("\nPrimeras grabaciones:")
display(recording_summary.head(10))

Guardar como csv

In [ ]:
summary_manifest = {
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "database_path": str(
        Path(db_path).resolve()
    ),

    "model_path": str(
        Path(model_path).resolve()
    ),

    "target_label": TARGET_LABEL,

    "classifier_classes": list(
        custom_classifier.classes
    ),

    "batch_size": BATCH_SIZE,

    "top_k_windows": TOP_K_WINDOWS,

    "reference_logit": REFERENCE_LOGIT,

    "n_windows": int(
        len(window_predictions)
    ),

    "n_recordings": int(
        len(recording_summary)
    ),

    "selection_score": SELECTION_SCORE,
}


manifest_path = (
    output_recording_summary
    .with_suffix(".json")
)


with open(
    manifest_path,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        summary_manifest,
        file,
        indent=2,
        ensure_ascii=False,
    )


print(
    "Manifest guardado en:",
    manifest_path,
)

In [ ]:
from pathlib import Path

pred_name = "pred_inicial"

model_name = Path(model_path).stem
model_dir = Path(model_path).parent
print(f'Nombre del modelo a utilizar es {model_name}, directorio del modelo: {model_dir} ')

output_csv = os.path.join(model_dir, f"{model_name}_{pred_name}.csv")
print(f'La prediccion se guardará en: {output_csv}' ) # ruta al archivo de salida de la prediccion

##### Ejecutar la inferencia

Aqui se puede configurar la sensibilidad en logit_threshold que define cual sera el umbral de score. Un logit_threshold = 0 significa que se guardaran solo las predicciones con score mayor a 0 (logit scale), este se puede utilizar para reducir la cantidad de predicciones a solo las consideradas positivas. Un logit_threshold = -np.inf se puede utilizar para incluir las predicciones para TODAS las ventanas de 5 s de la base de datos de audios. 


In [ ]:
# Ejecutar inferencia

#output_csv = Path(db_path) / "predicciones.csv"
logit_threshold = -np.inf
#logit_threshold = -0                       # ajusta según sensibilidad
labels = None                                   # o una tupla con etiquetas específica

output_csv = Path(output_csv)
output_csv.parent.mkdir(parents=True, exist_ok=True)

classifier.write_inference_csv(
    linear_classifier=custom_classifier,
    db=db,
    output_filepath=str(output_csv),
    threshold=logit_threshold,
    labels=labels,
    window_ids=db.match_window_ids()
)

print(f"\n✅ Inferencia completa. Resultados guardados en: {output_csv}")
